# ReShot — copy the shot, not the actors

Turn a reference video into a **depth map video** on a free Colab GPU, then give that grey clip to
Seedance or MiniMax H3 as the reference: it repeats the choreography and the camera moves with
your own characters in it.

**Before you run:** menu *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
Four steps: check the GPU · install · pick a clip · run and download.

Code and docs: [github.com/maosika-ai/reshot](https://github.com/maosika-ai/reshot) · Apache-2.0 ·
made at [Maosika 猫斯卡](https://www.maosika.com), the AI short-drama production system.

在免费的 Colab GPU 上把参考视频变成深度图视频，再交给 Seedance / MiniMax H3 当参考。
先在菜单 *代码执行程序 → 更改运行时类型* 选 **T4 GPU**，然后 *全部运行*。

## 1 · GPU

In [ ]:
#@title Check that this runtime has a GPU
import torch, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "no nvidia-smi")
print("torch", torch.__version__, "· CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then run again."

## 2 · Install

In [ ]:
#@title Install ReShot (about a minute; Colab already has a CUDA build of PyTorch)
in_china = False  #@param {type:"boolean"}
import os
if in_china:
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # the 111 MB weights come from Hugging Face
%pip install -q "reshot[ffmpeg]"
import reshot
print("reshot", reshot.__version__)

## 3 · Pick a clip

In [ ]:
#@title Use the demo clip, or upload your own (mp4 / mov / webm, best ≤ 15 s)
source = "demo clip (12 s corridor fight)"  #@param ["demo clip (12 s corridor fight)", "upload my own"]
import urllib.request, os
if source.startswith("upload"):
    from google.colab import files
    up = files.upload()
    assert up, "nothing uploaded"
    INPUT = next(iter(up))
else:
    INPUT = "city_night.mp4"
    if not os.path.exists(INPUT):
        urllib.request.urlretrieve("https://github.com/maosika-ai/reshot/raw/main/examples/city_night.mp4", INPUT)
import cv2
cap = cv2.VideoCapture(INPUT); n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
print(f"{INPUT}: {int(cap.get(3))}×{int(cap.get(4))} · {fps:.0f} fps · {n/fps:.1f} s"); cap.release()

## 4 · Make the depth map

In [ ]:
#@title Run (a 12 s clip takes ~40 s on a T4 at `fast`, ~2 min at `full`)
target = "seedance"  #@param ["seedance", "h3", "wan", "none"]
quality = "fast"  #@param ["fast", "full"]
import json, subprocess, sys, time
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "reshot", INPUT, "-o", "depth.mp4",
                    "--target", target, "--quality", quality, "--metrics", "metrics.json"],
                   capture_output=True, text=True)
print(r.stderr[-3000:])
assert r.returncode == 0 and os.path.exists("depth.mp4"), "run failed — read the message above"
m = json.load(open("metrics.json"))
print(f"done in {time.time()-t0:.0f} s · {m['frames']} frames @ {m['fps']} fps · "
      f"{m['output_size'][0]}×{m['output_size'][1]} · {m['ms_per_frame']:.0f} ms/frame on {m['device']}")

In [ ]:
#@title Look at it: reference and depth side by side, then the two clips
import base64, cv2
from IPython.display import HTML, display
def frame(path, idx=0):
    cap = cv2.VideoCapture(path); cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, bgr = cap.read(); cap.release()
    h = 360; bgr = cv2.resize(bgr, (int(bgr.shape[1] * h / bgr.shape[0]), h))
    return base64.b64encode(cv2.imencode(".jpg", bgr)[1]).decode()
def video(path):
    return f'<video controls loop muted autoplay width="360" src="data:video/mp4;base64,{base64.b64encode(open(path,"rb").read()).decode()}"></video>'
display(HTML(f'<div style="display:flex;gap:12px"><img src="data:image/jpeg;base64,{frame(INPUT)}"><img src="data:image/jpeg;base64,{frame("depth.mp4")}"></div>'
             f'<div style="display:flex;gap:12px;margin-top:12px">{video(INPUT)}{video("depth.mp4")}</div>'))

## 5 · Download, and what to do with it

In [ ]:
#@title Download depth.mp4 and print the prompt line for your video model
from google.colab import files
NEXT = {
  "seedance": ("Upload depth.mp4 to Seedance 2.0 / 2.5 as a reference video and start the prompt with:",
               "参考@视频1的动作与运镜，顺序与视频保持一致。\n(then describe your people, wardrobe, set and look)"),
  "h3": ("In MiniMax H3 attach depth.mp4 as <Video 1> (a character sheet as <Picture 1>); define it and say the grey look is not to be copied:",
         "<Subject 3> is the choreography and camera movement shown in <Video 1>, a grey depth map in which near objects are white and far objects are black: (one sentence on what happens in the clip)\n"
         "<Subject 3>: attribute_transfer - every action, position, timing and camera move of <Video 1> is transferred onto <Subject 1>; its grey depth look is not transferred."),
  "wan": ("Connect depth.mp4 to WanVaceToVideo → control_video in ComfyUI.", ""),
  "none": ("Hand depth.mp4 to any model that reads a depth video.", ""),
}
how, prompt = NEXT[target]
print(how); print(); print(prompt)
files.download("depth.mp4")

---
Made something? Post the reference, the depth map, the take and your prompt line in
[Show and tell](https://github.com/maosika-ai/reshot/discussions/2). Something broke?
[Open an issue](https://github.com/maosika-ai/reshot/issues/new/choose) with the message from step 4.

For whole folders, longer clips and scripting, run it on your own machine: `pip install reshot`,
then `reshot` for the local web page or `reshot clip.mp4 -o depth.mp4 --target seedance`.